# Customer Support RAG — Xumo Stream Box

Ye notebook `xumo_stream_box_guide.pdf` (setup, activation, troubleshooting, account & internet connectivity guide) par ek **Retrieval-Augmented Generation (RAG) based customer support bot** banata hai.

**Pipeline:**
`PDF Loader -> Text Splitter -> Open Source Embeddings -> FAISS Vector Store -> Retriever -> Prompt Template -> LLM -> Structured Output`

**Stack (LangChain v1, current as of Sep 2026):**
- `langchain-community` — `PyPDFLoader`, `FAISS`
- `langchain-text-splitters` — `RecursiveCharacterTextSplitter`
- `langchain-huggingface` — open source embeddings (`all-MiniLM-L6-v2`, no paid key)
- `langchain-groq` via `init_chat_model` — LLM for answer generation
- `langchain-core` — prompts, output parser, runnables, structured output



## 0. Setup — Install, Imports, PDF Path

In [1]:
!pip install -q -U langchain langchain-core langchain-text-splitters langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")  # community deprecation notice — harmless, package still works

from getpass import getpass

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain.chat_models import init_chat_model

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from pydantic import BaseModel, Field
from typing import Literal

print("All imports successful ✅")

All imports successful ✅


In [3]:
# Path to the Xumo Stream Box PDF.
# Notebook ko PDF ke saath usi folder me rakho, ya yaha poora path daal do
# (jaise: "/Users/pradipwasre/Desktop/GenAI-V2/xumo_stream_box_guide.pdf")
PDF_PATH = "xumo_stream_box_guide.pdf"

assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH} — update PDF_PATH above."
print("PDF found:", PDF_PATH)

PDF found: xumo_stream_box_guide.pdf


In [4]:
# Groq API key (free tier: https://console.groq.com/keys)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0)
print(llm.invoke("Say hello in one short line").content)

Hello!


## 1. Document Loader — Load the Xumo PDF

`PyPDFLoader` PDF ke har page ko ek alag `Document` object me load karta hai, page number metadata ke saath — isse baad me answer ke saath source page bhi bata sakte hain.

In [5]:
loader = PyPDFLoader(PDF_PATH)
pdf_docs = loader.load()

print("Total pages loaded:", len(pdf_docs))
print("\nPage 1 preview:\n", pdf_docs[0].page_content[:300])
print("\nMetadata of page 1:", pdf_docs[0].metadata)

Total pages loaded: 10

Page 1 preview:
 Xumo Stream Box
Complete Setup, Account & Troubleshooting Guide
This guide combines everything you need to set up, activate, and troubleshoot your Xumo Stream Box,
including how to sign in with your internet provider and manage your Xumo account.

Metadata of page 1: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-03T11:03:20+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-03T11:03:20+00:00', 'subject': '(unspecified)', 'title': 'Xumo Stream Box - Complete Setup & Support Guide', 'trapped': '/False', 'source': 'xumo_stream_box_guide.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}


## 2. Text Splitter


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

chunks = text_splitter.split_documents(pdf_docs)
print("Total chunks created:", len(chunks))
print("\nSample chunk:\n", chunks[5].page_content)
print("\nSource page:", chunks[5].metadata.get("page_label"))

Total chunks created: 23

Sample chunk:
 Get to Know Your Home Screen
Once activation is complete, you can navigate your home screen using rows such as Recommendations. The
Currently Playing tile features live content from your TV provider or Xumo Play. Navigate left to see recently
watched apps, or right for curated recommendations based on your watch history.

Source page: 3


## 3. Embeddings — Open Source Model

`sentence-transformers/all-MiniLM-L6-v2` — fast, lightweight, open source embedding model. Koi API key nahi chahiye, pehli baar chalane par model download hota hai.

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_vector = embedding_model.embed_query("How do I fix my remote?")
print("Embedding dimension:", len(test_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5681.17it/s]


Embedding dimension: 384


## 4. Vector Store — FAISS



In [8]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("xumo_faiss_index")

print("FAISS index built with", vectorstore.index.ntotal, "vectors")
print("Saved to ./xumo_faiss_index")

FAISS index built with 23 vectors
Saved to ./xumo_faiss_index


In [9]:
# Reload check — confirms the saved index works independently of the build step
vectorstore = FAISS.load_local(
    "xumo_faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True,
)
print("Reloaded index, total vectors:", vectorstore.index.ntotal)

Reloaded index, total vectors: 23


## 5. Retriever



In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

test_results = retriever.invoke("activation code not working")
for i, doc in enumerate(test_results):
    print(f"--- Match {i+1} (page {doc.metadata.get('page_label')}) ---")
    print(doc.page_content[:200])
    print()

--- Match 1 (page 4) ---
2. Troubleshooting Guide
Use these solutions to address activation code errors, connection problems, software update delays, and other
technical issues.
Why is my activation code not working?
Make sur

--- Match 2 (page 9) ---
5. Connect Your Xumo Stream Box to the Internet
A reliable internet connection lets you enjoy streaming without interruptions.
Connect During Activation
You'll be prompted to connect to the internet d

--- Match 3 (page 7) ---
1
If still no code: select "Skip & Set up later," confirm skipping, select the Xumo tile, and repeat if necessary
1
If a code still doesn't appear, unplug the device, plug it back in, and repeat activ

--- Match 4 (page 6) ---
What if I do not want to log in with my provider?
Select "Do it myself" when prompted to pick your internet provider. Then link your account using the on-screen
QR code or by visiting xumo.com/activat



## 6. Prompt Template — Customer Support Persona


In [11]:
support_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Xumo Stream Box customer support agent. "
     "Answer the customer's question using ONLY the context below. "
     "Be concise, friendly, and give clear step-by-step instructions when relevant. "
     "If the answer is not in the context, say you don't have that information and "
     "suggest contacting Xumo support via Start Chat — do not make anything up.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[Page {d.metadata.get('page_label')}] {d.page_content}" for d in docs
    )

## 7. Structured Output


In [12]:
class SupportResponse(BaseModel):
    """Structured customer support response for a Xumo Stream Box query."""
    answer: str = Field(description="Clear, step-by-step answer to the customer's question")
    category: Literal[
        "activation", "wifi_connectivity", "remote_issue", "account_login",
        "hardware_issue", "software_update", "other"
    ] = Field(description="Best matching support category for this query")
    needs_human_escalation: bool = Field(
        description="True if the answer was not found in context and a human agent should step in"
    )

structured_llm = llm.with_structured_output(SupportResponse)

structured_prompt = support_prompt  # same context+question format, different terminal step

def build_structured_chain():
    return (
        RunnableParallel(
            context=retriever | RunnableLambda(format_docs),
            question=RunnablePassthrough(),
        )
        | structured_prompt
        | structured_llm
    )

structured_chain = build_structured_chain()

## 8. Output Parser + LLM — Plain-Text RAG Chain


In [13]:
rag_chain = (
    RunnableParallel(
        context=retriever | RunnableLambda(format_docs),
        question=RunnablePassthrough(),
    )
    | support_prompt
    | llm
    | StrOutputParser()
)

## 9. Test the Xumo Support Bot



In [14]:
test_questions = [
    "My activation code is not working, what should I do?",
    "The screen is black, how do I fix it?",
    "How do I connect my device to a new WiFi network?",
    "Do I need a Xumo account to use the Stream Box?",
    "What is the capital of France?",  # out-of-scope question to test guardrails
]

for q in test_questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print("-" * 80)

Q: My activation code is not working, what should I do?
A: Sure! Here’s what to do when your activation code isn’t working:

1. **Verify the code**  
   - Make sure you’re entering the 6‑digit code exactly as shown on the screen.  
   - Go to the correct web address: **xumo.com/activate**.

2. **Power‑cycle the box to get a new code**  
   1. Unplug the power cable from your Xumo Stream Box.  
   2. Unplug the power cable from your TV.  
   3. Wait **15 seconds**.  
   4. Plug both power cables back in.  

3. **Enter the new code**  
   - After the box restarts, return to **xumo.com/activate** and type the new 6‑digit code (remember, each code is only valid for 10 minutes).

4. **If it still won’t work**  
   - Contact Xumo customer support via **Start Chat** for further assistance.  

Hope that helps! Let me know if you need anything else.
--------------------------------------------------------------------------------
Q: The screen is black, how do I fix it?
A: Sure! A black screen u

In [15]:
# Same questions through the structured output version
for q in test_questions[:3]:
    result = structured_chain.invoke(q)
    print("Q:", q)
    print(result)
    print("-" * 80)

Q: My activation code is not working, what should I do?
answer='1. Make sure you’re entering the 6‑digit activation code shown on your TV screen at **xumo.com/activate**.\n2. If it still won’t work, power‑cycle the box to get a new code:\n   - Unplug the power cable from the Xumo Stream Box.\n   - Unplug the power cable from your TV.\n   - Wait about 15 seconds.\n   - Plug both power cables back in.\n3. When the Stream Box restarts, go back to **xumo.com/activate** and enter the new code (each code is only valid for 10 minutes).\n4. If you still can’t get a working code, contact Xumo customer support via **Start Chat**.' category='activation' needs_human_escalation=False
--------------------------------------------------------------------------------
Q: The screen is black, how do I fix it?
answer='A black screen usually means the box is in standby mode.\n1. Press the Home button on your Xumo remote once (just a quick press, don’t hold it). Make sure the remote is close to the box with

## 10. Interactive Support Chat (Optional)

In [16]:
def ask_xumo_support(query: str) -> str:
    """Simple helper function to query the Xumo support RAG chain."""
    return rag_chain.invoke(query)

# Example single call:
# print(ask_xumo_support("My remote is not responding, how do I fix it?"))

while True:
    user_query = input("Ask Xumo Support (or type 'exit'): ")
    if user_query.strip().lower() in ("exit", "quit"):
        print("Chat ended.")
        break
    print("Bot:", ask_xumo_support(user_query))
    print()

Bot: You can locate the MAC address right on the box itself or in the system menu, depending on your model:

**ES1 Xumo Stream Box**  
- Turn the box over.  
- The MAC address is printed on the bottom of the device and is labeled **“MAC.”**  
- It looks like XX:XX:XX:XX:XX:XX.

**EntOS Xumo Stream Box**  
1. Open **Settings**.  
2. Select **System management**.  
3. Choose **System info**.  
4. The **MAC address (Ethernet)** will be displayed there.

If you need any more help, just let me know!

Bot: I’m sorry, but I don’t have enough information to answer that question. Please contact Xumo support via **Start Chat** for further assistance.

Chat ended.


### Recap

| Step | Component | Notes |
|---|---|---|
| Load | `PyPDFLoader` | 1 Document per PDF page |
| Split | `RecursiveCharacterTextSplitter` | 800 char chunks, 120 overlap |
| Embed | `HuggingFaceEmbeddings` | open source, `all-MiniLM-L6-v2` |
| Store | `FAISS` | saved locally to `./xumo_faiss_index` |
| Retrieve | `vectorstore.as_retriever()` | top-4 similarity search |
| Generate | `init_chat_model` (Groq) | grounded, context-only answers |
| Structured Output | `with_structured_output` | category + escalation flag for CRM use |
| Output Parser | `StrOutputParser` | plain text for chatbot widgets |

